The purpose of this notebook is to verify the correctness of the functions in `utils_qubit_hamiltonian.py` and `utils_states.py`, and present simple examples of their usage

In [1]:
import sys
sys.path.append('../')

import numpy as np
from utils_hamiltonian_qubit import (
    split_pauli_operator,
    pauli_matrix_element_with_basis_state,
    remove_irrelevant_pauli_terms,
    taper_hamiltonian,
    CNOT_clifford_operator,
    seniority_solving_clifford_operator,
    append_seniority_solving_clifford_circuit,
    group_odds_and_evens,
    ungroup_odds_and_evens,
    process_qubit_hamiltonian_to_remove_irrelevant_terms,
    process_qubit_hamiltonian_to_project_onto_symmetric_subspace
)
from openfermion import (
    QubitOperator,
    get_sparse_operator,
    get_fermion_operator,
    MolecularData,
    jordan_wigner
)
from openfermionpyscf import run_pyscf
from utils_basic import (
    random_pauli_term, 
    random_bin_list, 
    state_from_bin_list,
    compute_product_of_unitaries,
    random_pauli_hamiltonian,
    apply_unitary_product,
    compute_product_of_unitaries
)
from utils_misc import (
    duplicate_pauli_term_qubit_wise,
    duplicate_hamiltonian_qubit_wise,
    duplicate_statevector_qubit_wise,
    simple_print_state
)
from utils_fc import (
    decimal_to_binary_list,
    decimal_to_binary_string
)
from utils_states import (
    create_composite_state,
    somos_to_seniority_config,
    compress_state,
    enlarge_binary_index_using_config,
    decompress_state
)

from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator
from numpy.random import uniform
import random

## Pauli Operator Matrix Elements

Here, I verify the following:
- `split_pauli_operator` should cleave a Pauli operator into two Pauli operators correctly


In [ ]:
P = QubitOperator('X0 X1 X2 X3 X4 Y5 Y6 Y7 Y8 Y9')
N = 5
P0, P1 = split_pauli_operator(P, 5)

print(f'''
    Original Pauli : {P}
    Left Split     : {QubitOperator(P0)}
    Right Split    : {QubitOperator(P1)}
''')


P = QubitOperator('X0 Y5 Y6 Y7 Y8 Y9')
N = 3
P0, P1 = split_pauli_operator(P, 5)

print(f'''
    Original Pauli : {P}
    Left Split     : {QubitOperator(P0)}
    Right Split    : {QubitOperator(P1)}
''')

Here, I verify the following:

- <v|P|w> is calculated correctly for a bunch of P, v, w examples by comparing with numerical value

In [ ]:
for Nqubits in range(2, 12):
    for _ in range(40):
        _, P = random_pauli_term(Nqubits)
        v    = random_bin_list(Nqubits)
        w    = random_bin_list(Nqubits)

        matrix_element_numerical  = state_from_bin_list(v) @ get_sparse_operator(P, Nqubits) @ state_from_bin_list(w)
        matrix_element_analytical = pauli_matrix_element_with_basis_state(P, v, w)
        
        print(np.abs(matrix_element_numerical - matrix_element_analytical) < 1e-12, end='')
    print()

## Hamiltonian modification based on symmetries

First, I generate a simple non-trivial Hamiltonian to work with, and implement the term deletion and tapering based on computational basis states.

In [ ]:
geo = [
    ['H', [0,0,0]],
    ['H', [0,0,1]],
    ['H', [0,0,2]],
    ['H', [0,0,3]]
]
mol = run_pyscf(
    MolecularData(geo, 'sto3g', 1, 0), 
    run_scf=True
)
H = jordan_wigner(get_fermion_operator(mol.get_molecular_hamiltonian()))

# play around with this example by modifying v and w
v = [1,0,0,1]
w = [0,1,1,0]

Hrel     = remove_irrelevant_pauli_terms(H, v, w)
Htapered = taper_hamiltonian(H, v, w)

print(f'''
    Number of terms in H        : {len(H.terms)}
    Number of terms in Hrel     : {len(Hrel.terms)}
    Number of terms in Htapered : {len(Htapered.terms)}
''')

print('Hrel')
print(Hrel)
print()

print('Htapered')
print(Htapered)

print()
print(Htapered - taper_hamiltonian(Hrel, v, w))


The purpose of this code is to modify the Hamiltonian without changing the relevant matrix element and/or expectation values. So we should see if that works as desired.

Here, I see if, for states $|\Psi_s\rangle = |s\rangle |\psi_s\rangle$, and $|\Psi_t\rangle = |t\rangle |\psi_t\rangle$, we have

$$
\langle \Psi_s|H|\Psi_t\rangle = \langle \Psi_s|H_{\text{rel}}|\Psi_t\rangle = \langle \psi_s|H_\text{tapered}|\psi_t \rangle
$$

I will use the LiH Hamiltonian since it is larger

In [ ]:
geo = [
    ['H', [0,0,0]],
    ['Li', [0,0,1]]
]
mol     = run_pyscf(MolecularData(geo, 'sto3g', 1, 0), run_scf=True)
H       = jordan_wigner(get_fermion_operator(mol.get_molecular_hamiltonian()))
Nqubits = 12

for _ in range(20):
    taper_size = random.sample(range(2, 8), 1)[0]

    bin_s = random_bin_list(taper_size)
    bin_t = random_bin_list(taper_size)

    ket_s = state_from_bin_list(bin_s, taper_size)
    ket_t = state_from_bin_list(bin_t, taper_size)

    psi_s = uniform(-1, 1, 2 ** (Nqubits - taper_size))
    psi_t = uniform(-1, 1, 2 ** (Nqubits - taper_size))

    Psi_s = np.kron(ket_s, psi_s)
    Psi_t = np.kron(ket_t, psi_t)

    Hsparse         = get_sparse_operator(H, Nqubits)
    ev_full         = Psi_s @ Hsparse @ Psi_t

    Hrel_sparse     = get_sparse_operator(remove_irrelevant_pauli_terms(H, bin_s, bin_t), Nqubits)
    ev_rel          = Psi_s @ Hrel_sparse @ Psi_t

    Htapered_sparse = get_sparse_operator(taper_hamiltonian(H, bin_s, bin_t), Nqubits - taper_size)
    ev_tapered      = psi_s @ Htapered_sparse @ psi_t

    print(f'''
        full expectation    : {ev_full}
        rel expectation     : {ev_rel}
        tapered expectation : {ev_tapered}

        all equal           : {np.abs(ev_full-ev_rel)<1e-12 and np.abs(ev_full-ev_tapered)<1e-12 and np.abs(ev_rel-ev_tapered)<1e-12}
    ''')

## Solving Seniority Operators

First, I check that $ e^{-i\pi/4} U_\text{CNOT}(0,1) = \text{CNOT}(0,1)$ and $e^{-i \pi / 4} U_\text{CNOT}(1,0) = \text{CNOT}(1,0)$

In [ ]:
CNOT_01 = np.array([
    [1,0,0,0],
    [0,1,0,0],
    [0,0,0,1],
    [0,0,1,0]
])

CNOT_10 = np.array([
    [1,0,0,0],
    [0,0,0,1],
    [0,0,1,0],
    [0,1,0,0]
])

CNOT_01_from_func = get_sparse_operator(compute_product_of_unitaries(CNOT_clifford_operator(0,1)), 2).toarray()

CNOT_10_from_func = get_sparse_operator(compute_product_of_unitaries(CNOT_clifford_operator(1,0)), 2).toarray()


print(
    np.allclose(np.exp(-1j * np.pi / 4) * CNOT_01_from_func, CNOT_01), end=''
)

print(
    np.allclose(np.exp(-1j * np.pi / 4) * CNOT_10_from_func, CNOT_10)
)

Now I check if the seniority solving operator and the seniority solving circuits are the same

In [ ]:
for Nqubits in [2,4,6,8,10,12]:
    Uof = get_sparse_operator(compute_product_of_unitaries(seniority_solving_clifford_operator(Nqubits)), Nqubits).toarray()

    qc = QuantumCircuit(Nqubits)
    append_seniority_solving_clifford_circuit(qc, Nqubits)

    Uqk = Operator(qc).reverse_qargs().data

    print(
        np.allclose(np.exp(-1j * np.pi * (Nqubits // 2) / 4) * Uof, Uqk), end=''
    )

Now, I check if $U z_{2i} z_{2i+1} U^T = z_{2i}$

In [ ]:
Nqubits = 12
Ucliff  = seniority_solving_clifford_operator(Nqubits)

for i in range(Nqubits // 2):
    parity  = QubitOperator(f'Z{2*i} Z{2*i+1}')
    rotated = apply_unitary_product(parity, Ucliff)
    print(rotated == QubitOperator(f'Z{2*i}'), end='')

## Full Tapering Procedure for Seniority

First: I check manually the reordering maps. First: applying both should produce the Hamiltonian I started with.

In [ ]:
for Nqubits in range(2, 12):
    for Nterms in range(Nqubits // 2, 5 * Nqubits):
        H       = random_pauli_hamiltonian(Nqubits, Nterms)
        Hrecov1 = ungroup_odds_and_evens(group_odds_and_evens(H, Nqubits), Nqubits)
        Hrecov2 = group_odds_and_evens(ungroup_odds_and_evens(H, Nqubits), Nqubits)

        print(H - Hrecov1, end='')
        print(H - Hrecov2, end='')
        print(Hrecov1 - Hrecov2, end='')
    print()

Next, I check the reordering maps on some simple examples.

In [ ]:
H          = QubitOperator('X0 Y1 X2 Y3 X4 Y5')
Hgrouped   = group_odds_and_evens(H, Nqubits)
Hungrouped = ungroup_odds_and_evens(Hgrouped, Nqubits)
Nqubits    = 6

print(f'''
    H          : {list(H.terms.keys())}
    Hgrouped   : {list(Hgrouped.terms.keys())}
    Hungrouped : {list(Hungrouped.terms.keys())}

    {H - Hungrouped == QubitOperator().zero()}
''')

Next, I check if, for a seniority conserving Hamiltonian, applying the Clifford transformation and regrouping produces zs on the first N/2 qubits

In [ ]:
Nqubits = 12

H = duplicate_hamiltonian_qubit_wise(
    np.round(uniform(-2, 2), 4) * QubitOperator('X0 Y1 X2 Y3 X4 X5') + 
    np.round(uniform(-2, 2), 4) * QubitOperator('Y0 X1 Z2 X3 Z4 Y5') + 
    np.round(uniform(-2, 2), 4) * QubitOperator('Y0 Y1 X2 X3 X4 X5') + 
    np.round(uniform(-2, 2), 4) * QubitOperator('X0 X1 Z2 Y3 Z4 Z5') + 
    np.round(uniform(-2, 2), 4) * QubitOperator('Z0 X1 X2 Y3 X4 Y5') + 
    np.round(uniform(-2, 2), 4) * QubitOperator('Z0 Y1 Z2 Z3 Z4 Y5') + 
    np.round(uniform(-2, 2), 4) * QubitOperator('X0 Z1 X2 X3 Z4 Y5')
)

Ucliff = seniority_solving_clifford_operator(Nqubits)

Hrotated_temp           = apply_unitary_product(H, Ucliff)

Hrotated = QubitOperator()
for term, coef in Hrotated_temp.terms.items():
    Hrotated += np.round(coef, 4) * QubitOperator(term)

Hrotated_reordered = group_odds_and_evens(Hrotated, Nqubits)

print(H)

print()

print(Hrotated)

print()

print(Hrotated_reordered)

It is instructive to see how it maps all seniority conserving operators.

In [ ]:
XX = QubitOperator('X0 X1')
YY = QubitOperator('Y0 Y1')
ZZ = QubitOperator('Z0 Z1')
II = QubitOperator('')
XY = QubitOperator('X0 Y1')
YX = QubitOperator('Y0 X1')
ZI = QubitOperator('Z0')
IZ = QubitOperator('Z1')

Ucliff = seniority_solving_clifford_operator(Nqubits=2)

print(apply_unitary_product(XX, Ucliff))
print(apply_unitary_product(YY, Ucliff))
print(apply_unitary_product(ZZ, Ucliff))
print(apply_unitary_product(II, Ucliff))
print(apply_unitary_product(XY, Ucliff))
print(apply_unitary_product(YX, Ucliff))
print(apply_unitary_product(ZI, Ucliff))
print(apply_unitary_product(IZ, Ucliff))

Next, I check if the seniority solving Clifford transformation correctly encodes the seniority configuration on the even-index qubits for a 2N qubit state

In [ ]:
# seniority zero state

Nqubits     = 8

psi_source  = uniform(-1, 1, 2 ** (Nqubits // 2))
psi_source  = psi_source / np.linalg.norm(psi_source)
psi_full    = duplicate_statevector_qubit_wise(psi_source)

Ucliff      = get_sparse_operator(compute_product_of_unitaries(seniority_solving_clifford_operator(Nqubits)), Nqubits)

psi_tapered = Ucliff @ psi_full

RED         = '\033[91m'
DEFAULT     = '\033[0m'


for i, val in enumerate(psi_tapered):
    if np.abs(val) > 1e-12:
        bin_list            = decimal_to_binary_list(i, length=Nqubits)
        bin_string          = ''.join([str(z) for z in bin_list])
        coloured_bin_string = ''
        for i, s in enumerate(bin_string):
            if i % 2 == 0:
                coloured_bin_string += f"{RED}{s}{DEFAULT}"
            else:
                coloured_bin_string += s

        print(coloured_bin_string, np.round(val, 4))


In [ ]:
# seniority [0,1,0,0] state

Nqubits     = 8

first       = duplicate_statevector_qubit_wise(uniform(-1, 1, 2))
second      = uniform(-1, 1) * np.array([0,1,0,0]) + uniform(-1, 1) * np.array([0,0,1,0])
third       = duplicate_statevector_qubit_wise(uniform(-1, 1, 4))

psi_full    = np.kron(np.kron(first, second), third)

Ucliff      = get_sparse_operator(compute_product_of_unitaries(seniority_solving_clifford_operator(Nqubits)), Nqubits)

psi_tapered = Ucliff @ psi_full

RED         = '\033[91m'
DEFAULT     = '\033[0m'


for i, val in enumerate(psi_tapered):
    if np.abs(val) > 1e-12:
        bin_list            = decimal_to_binary_list(i, length=Nqubits)
        bin_string          = ''.join([str(z) for z in bin_list])
        coloured_bin_string = ''
        for i, s in enumerate(bin_string):
            if i % 2 == 0:
                coloured_bin_string += f"{RED}{s}{DEFAULT}"
            else:
                coloured_bin_string += s

        print(coloured_bin_string, np.round(val, 4))


Now, I focus on the two main functions of `utils_hamiltonian.py`, which process the Hamiltonian into a modified Hamiltonian for quantum measurements.

The first check is: the matrix element of two seniority zero states is unmodified by the processing of the Hamiltonian

In [ ]:
for Nqubits in range(2, 15, 2):
    Nterms      = 1000
    zero_config = [0 for _ in range(Nqubits // 2)]

    # generate random bra and ket

    ket = duplicate_statevector_qubit_wise(uniform(-1, 1, 2 ** (Nqubits // 2)))
    ket = ket / np.linalg.norm(ket)

    bra = duplicate_statevector_qubit_wise(uniform(-1, 1, 2 ** (Nqubits // 2)))
    bra = bra / np.linalg.norm(bra)

    # generate random Hamiltonian and process

    H                 = random_pauli_hamiltonian(Nqubits, Nterms)
    H_sparse          = get_sparse_operator(H, Nqubits)

    Hprocessed        = process_qubit_hamiltonian_to_remove_irrelevant_terms(H, Nqubits, zero_config, zero_config)
    Hprocessed_sparse = get_sparse_operator(Hprocessed, Nqubits)

    print(f'''
        Number of qubits              = {Nqubits}
        Number of terms in H          = {len(H.terms)}
        Number of terms in Hprocessed = {len(Hprocessed.terms)}
        <bra|H|ket>                   = {np.round(bra @ H_sparse @ ket, 6)}
        <bra|Hprocessed|ket>          = {np.round(bra @ Hprocessed_sparse @ ket, 6)}
    ''')

Now I repeat the same check for seniority non-zero states. Because these are harder to generate as a black-box I will do each example manually

In [ ]:
Nqubits    = 8
bra_config = [0,0,0,0]
ket_config = [0,1,0,0]

bra = duplicate_statevector_qubit_wise(uniform(-1, 1, 2 ** (Nqubits // 2)))
bra = bra / np.linalg.norm(bra)

first  = duplicate_statevector_qubit_wise(uniform(-1, 1, 2))
second = uniform(-1, 1) * np.array([0,1,0,0]) + uniform(-1, 1) * np.array([0,0,1,0])
third  = duplicate_statevector_qubit_wise(uniform(-1, 1, 4))

ket = np.kron(np.kron(first, second), third)
ket = ket / np.linalg.norm(ket)

# generate random Hamiltonian and process

H                 = random_pauli_hamiltonian(Nqubits, Nterms)
H_sparse          = get_sparse_operator(H, Nqubits)

Hprocessed        = process_qubit_hamiltonian_to_remove_irrelevant_terms(H, Nqubits, bra_config, ket_config)
Hprocessed_sparse = get_sparse_operator(Hprocessed, Nqubits)

print(f'''
    Number of qubits              = {Nqubits}
    Number of terms in H          = {len(H.terms)}
    Number of terms in Hprocessed = {len(Hprocessed.terms)}
    <bra|H|ket>                   = {np.round(bra @ H_sparse @ ket, 6)}
    <bra|Hprocessed|ket>          = {np.round(bra @ Hprocessed_sparse @ ket, 6)}
''')

In [ ]:
Nqubits    = 12
bra_config = [0,0,0,0,0,0]
ket_config = [0,1,0,0,0,0]

bra = duplicate_statevector_qubit_wise(uniform(-1, 1, 2 ** (Nqubits // 2)))
bra = bra / np.linalg.norm(bra)

first  = duplicate_statevector_qubit_wise(uniform(-1, 1, 2))
second = uniform(-1, 1) * np.array([0,1,0,0]) + uniform(-1, 1) * np.array([0,0,1,0])
third  = duplicate_statevector_qubit_wise(uniform(-1, 1, 2**4))

ket = np.kron(np.kron(first, second), third)
ket = ket / np.linalg.norm(ket)

# generate random Hamiltonian and process

H                 = random_pauli_hamiltonian(Nqubits, Nterms)
H_sparse          = get_sparse_operator(H, Nqubits)

Hprocessed        = process_qubit_hamiltonian_to_remove_irrelevant_terms(H, Nqubits, bra_config, ket_config)
Hprocessed_sparse = get_sparse_operator(Hprocessed, Nqubits)

print(f'''
    Number of qubits              = {Nqubits}
    Number of terms in H          = {len(H.terms)}
    Number of terms in Hprocessed = {len(Hprocessed.terms)}
    <bra|H|ket>                   = {np.round(bra @ H_sparse @ ket, 6)}
    <bra|Hprocessed|ket>          = {np.round(bra @ Hprocessed_sparse @ ket, 6)}
''')

In [ ]:
Nqubits    = 12
bra_config = [0,0,0,1,1,0]
ket_config = [0,1,1,0,0,0]

first  = duplicate_statevector_qubit_wise(uniform(-1, 1, 2**3))
second = uniform(-1, 1) * np.array([0,1,0,0]) + uniform(-1, 1) * np.array([0,0,1,0])
third  = uniform(-1, 1) * np.array([0,1,0,0]) + uniform(-1, 1) * np.array([0,0,1,0])
fourth = duplicate_statevector_qubit_wise(uniform(-1, 1, 2**1))

bra = np.kron(np.kron(np.kron(first, second), third), fourth)
bra = bra / np.linalg.norm(bra)

first  = duplicate_statevector_qubit_wise(uniform(-1, 1, 2**1))
second = uniform(-1, 1) * np.array([0,1,0,0]) + uniform(-1, 1) * np.array([0,0,1,0])
third  = uniform(-1, 1) * np.array([0,1,0,0]) + uniform(-1, 1) * np.array([0,0,1,0])
fourth = duplicate_statevector_qubit_wise(uniform(-1, 1, 2**3))

ket = np.kron(np.kron(np.kron(first, second), third), fourth)
ket = ket / np.linalg.norm(ket)

# generate random Hamiltonian and process

H                 = random_pauli_hamiltonian(Nqubits, Nterms)
H_sparse          = get_sparse_operator(H, Nqubits)

Hprocessed        = process_qubit_hamiltonian_to_remove_irrelevant_terms(H, Nqubits, bra_config, ket_config)
Hprocessed_sparse = get_sparse_operator(Hprocessed, Nqubits)

print(f'''
    Number of qubits              = {Nqubits}
    Number of terms in H          = {len(H.terms)}
    Number of terms in Hprocessed = {len(Hprocessed.terms)}
    <bra|H|ket>                   = {np.round(bra @ H_sparse @ ket, 6)}
    <bra|Hprocessed|ket>          = {np.round(bra @ Hprocessed_sparse @ ket, 6)}
''')

Now, I also compare with the tapered Hamiltonian. So, below, I check if the original H, the processed H, and the tapered H, all have the same matrix elements for states with fixed seniority configurations. First, I check for zero seniority.

In [ ]:
for Nqubits in range(2, 15, 2):
    Nterms      = 1000
    zero_config = [0 for _ in range(Nqubits // 2)]

    bra_N  = uniform(-1, 1, 2 ** (Nqubits // 2))
    bra_N  = bra_N / np.linalg.norm(bra_N)
    bra_2N = duplicate_statevector_qubit_wise(bra_N)
    bra_2N = bra_2N / np.linalg.norm(bra_2N)

    ket_N  = uniform(-1, 1, 2 ** (Nqubits // 2))
    ket_N  = ket_N / np.linalg.norm(ket_N)
    ket_2N = duplicate_statevector_qubit_wise(ket_N)
    ket_2N = ket_2N / np.linalg.norm(ket_2N)

    H          = random_pauli_hamiltonian(Nqubits, Nterms)
    Hprocessed = process_qubit_hamiltonian_to_remove_irrelevant_terms(H, Nqubits, zero_config, zero_config)
    Htapered   = process_qubit_hamiltonian_to_project_onto_symmetric_subspace(H, Nqubits, zero_config, zero_config)

    H_sparse          = get_sparse_operator(H, Nqubits)
    Hprocessed_sparse = get_sparse_operator(Hprocessed, Nqubits)
    Htapered_sparse   = get_sparse_operator(Htapered, Nqubits // 2)


    print(f'''
        Number of qubits              = {Nqubits}
        Number of terms in H          = {len(H.terms)}
        Number of terms in Hprocessed = {len(Hprocessed.terms)}
        Number of terms in Htapered   = {len(Htapered.terms)}
        <bra|H|ket>                   = {np.round(bra_2N @ H_sparse @ ket_2N, 6)}
        <bra|Hprocessed|ket>          = {np.round(bra_2N @ Hprocessed_sparse @ ket_2N, 6)}
        <bra|Htapered|ket>            = {np.round(bra_N @ Htapered_sparse @ ket_N, 6)}
    ''')

In [ ]:
for _ in range(10):
    Nqubits     = 12
    Nterms      = 1000
    bra_config  = [0,1,0,0,0,0]
    ket_config  = [0,1,0,0,0,0]

    # make bra -> for now assume the ket is the same

    one_N    = uniform(-1, 1, 2**1)
    one_2N   = duplicate_statevector_qubit_wise(one_N)

    two_N    = uniform(-1, 1) * np.array([0,1]) + uniform(-1, 1) * np.array([1,0])
    two_2N   = two_N[0] * np.array([0,0,1,0]) + two_N[1] * np.array([0,1,0,0])

    three_N  = uniform(-1, 1, 2**4)
    three_2N = duplicate_statevector_qubit_wise(three_N)

    bra_N    = np.kron(np.kron(one_N, two_N), three_N)
    bra_2N   = np.kron(np.kron(one_2N, two_2N), three_2N)

    ket_N    = bra_N.copy()
    ket_2N   = bra_2N.copy()

    H          = random_pauli_hamiltonian(Nqubits, Nterms)
    Hprocessed = process_qubit_hamiltonian_to_remove_irrelevant_terms(H, Nqubits, bra_config, ket_config)
    Htapered   = process_qubit_hamiltonian_to_project_onto_symmetric_subspace(H, Nqubits, bra_config, ket_config)

    H_sparse          = get_sparse_operator(H, Nqubits)
    Hprocessed_sparse = get_sparse_operator(Hprocessed, Nqubits)
    Htapered_sparse   = get_sparse_operator(Htapered, Nqubits // 2)


    print(f'''
        Number of qubits              = {Nqubits}
        Number of terms in H          = {len(H.terms)}
        Number of terms in Hprocessed = {len(Hprocessed.terms)}
        Number of terms in Htapered   = {len(Htapered.terms)}
        <bra|H|ket>                   = {np.round(bra_2N @ H_sparse @ ket_2N, 6)}
        <bra|Hprocessed|ket>          = {np.round(bra_2N @ Hprocessed_sparse @ ket_2N, 6)}
        <bra|Htapered|ket>            = {np.round(bra_N @ Htapered_sparse @ ket_N, 6)}
    ''')

In [ ]:
for _ in range(10):
    Nqubits    = 12
    Nterms     = 5000
    bra_config = [0,0,1,1,0,0]
    ket_config = [0,1,0,1,0,0]

    one_N    = uniform(-1, 1, 2**2)
    one_2N   = duplicate_statevector_qubit_wise(one_N)

    two_N    = uniform(-1, 1) * np.array([0,1]) + uniform(-1, 1) * np.array([1,0])
    two_2N   = two_N[0] * np.array([0,0,1,0]) + two_N[1] * np.array([0,1,0,0])

    three_N  = uniform(-1, 1) * np.array([0,1]) + uniform(-1, 1) * np.array([1,0])
    three_2N = three_N[0] * np.array([0,0,1,0]) + three_N[1] * np.array([0,1,0,0])

    four_N   = uniform(-1, 1, 2**2)
    four_2N  = duplicate_statevector_qubit_wise(four_N)

    bra_N  = np.kron(np.kron(np.kron(one_N, two_N), three_N), four_N)
    bra_N  = bra_N / np.linalg.norm(bra_N)
    bra_2N = np.kron(np.kron(np.kron(one_2N, two_2N), three_2N), four_2N)
    bra_2N = bra_2N / np.linalg.norm(bra_2N)

    one_N      = uniform(-1, 1, 2**1)
    one_2N     = duplicate_statevector_qubit_wise(one_N)

    two_N      = uniform(-1, 1) * np.array([0,1]) + uniform(-1, 1) * np.array([1,0])
    two_2N     = two_N[0] * np.array([0,0,1,0]) + two_N[1] * np.array([0,1,0,0])

    three_N    = uniform(-1, 1, 2**1)
    three_2N   = duplicate_statevector_qubit_wise(three_N)

    four_N    = uniform(-1, 1) * np.array([0,1]) + uniform(-1, 1) * np.array([1,0])
    four_2N   = four_N[0] * np.array([0,0,1,0]) + four_N[1] * np.array([0,1,0,0])

    five_N    = uniform(-1, 1, 2**2)
    five_2N   = duplicate_statevector_qubit_wise(five_N)

    ket_N  = np.kron(np.kron(np.kron(np.kron(one_N, two_N), three_N), four_N), five_N)
    ket_N  = ket_N / np.linalg.norm(ket_N)
    ket_2N = np.kron(np.kron(np.kron(np.kron(one_2N, two_2N), three_2N), four_2N), five_2N)
    ket_2N = ket_2N / np.linalg.norm(ket_2N)

    H          = random_pauli_hamiltonian(Nqubits, Nterms)
    Hprocessed = process_qubit_hamiltonian_to_remove_irrelevant_terms(H, Nqubits, bra_config, ket_config)
    Htapered   = process_qubit_hamiltonian_to_project_onto_symmetric_subspace(H, Nqubits, bra_config, ket_config)

    H_sparse          = get_sparse_operator(H, Nqubits)
    Hprocessed_sparse = get_sparse_operator(Hprocessed, Nqubits)
    Htapered_sparse   = get_sparse_operator(Htapered, Nqubits // 2)

    print(f'''
        Number of qubits              = {Nqubits}
        Number of terms in H          = {len(H.terms)}
        Number of terms in Hprocessed = {len(Hprocessed.terms)}
        Number of terms in Htapered   = {len(Htapered.terms)}
        <bra|H|ket>                   = {np.round(bra_2N @ H_sparse @ ket_2N, 6)}
        <bra|Hprocessed|ket>          = {np.round(bra_2N @ Hprocessed_sparse @ ket_2N, 6)}
        <bra|Htapered|ket>            = {np.round(bra_N @ Htapered_sparse @ ket_N, 6)}
    ''')

## Tapering Procedure Including States

Here, I check if the functions which compress and decompress the states work. I start with a simple example, then move to a generic example.

In [ ]:
# example 1

Nqubits            = 6
alpha, beta, gamma = 1.0, 2.0, 3.0
config             = [0,1,0]

psi = np.zeros(2 ** Nqubits)
psi[int('111000', 2)] = alpha          # tapered index: 100
psi[int('000111', 2)] = beta           # tapered index: 011
psi[int('110100', 2)] = gamma          # tapered index: 110

simple_print_state(psi)
print()

psi_t = compress_state(psi)
simple_print_state(psi_t)
print()

psi_recov = decompress_state(psi_t, config)
simple_print_state(psi_recov)
print()


In [ ]:
Nqubits = 8

for config in [0,0,0,0], [0,0,1,0], [1,0,1,0], [1,1,0,0], [1,1,1,0], [1,0,1,1], [1,1,1,1]:
    for _ in range(8):
        psi_t     = uniform(-1, 1, 2 ** (Nqubits // 2))
        psi_t     = psi_t / np.linalg.norm(psi_t)
        psi_t_r   = compress_state(decompress_state(psi_t, config))
        psi_t_r_r = compress_state(decompress_state(psi_t_r, config))

        print(np.linalg.norm(psi_t - psi_t_r) == 0.0, end='')
        print(np.linalg.norm(psi_t - psi_t_r_r) == 0.0, end='')
        print(np.abs(np.linalg.norm(psi_t_r) - 1.0) < 1e-14, end='')
        print(np.abs(np.linalg.norm(psi_t_r_r) - 1.0) < 1e-14, end='')

    print()

In [ ]:
Nqubits = 12


for config in ([0,0,0,0,0,0], 
               [0,0,0,1,0,0], 
               [0,1,0,0,0,0], 
               [0,0,1,1,0,0], 
               [0,1,0,0,1,0], 
               [0,0,0,0,1,1], 
               [1,0,0,1,0,1], 
               [0,1,1,1,0,0],
               [0,0,1,1,0,1], 
               [0,1,1,0,1,0], 
               [1,0,0,1,0,1], 
               [0,1,1,1,1,0], 
               [1,1,0,1,1,0], 
               [1,1,1,0,1,0], 
               [1,1,1,0,1,1], 
               [1,1,1,1,1,1]):
    for _ in range(8):

        psi_t     = uniform(-1, 1, 2 ** (Nqubits // 2))
        psi_t     = psi_t / np.linalg.norm(psi_t)
        psi_t_r   = compress_state(decompress_state(psi_t, config))
        psi_t_r_r = compress_state(decompress_state(psi_t_r, config))

        print(np.linalg.norm(psi_t - psi_t_r) == 0.0, end='')
        print(np.linalg.norm(psi_t - psi_t_r_r) == 0.0, end='')
        print(np.abs(np.linalg.norm(psi_t_r) - 1.0) < 1e-14, end='')
        print(np.abs(np.linalg.norm(psi_t_r_r) - 1.0) < 1e-14, end='')

    print()

## Matrix elements with and without tapering

Let $|\phi\rangle$ and $|\psi\rangle$ denote two $2N$ qubit states with configurations $\vec{v}, \vec{w}$ respectively. Let $H$ denote the Hamiltonian, $H(\vec{v}, \vec{w})$ denote the part of the Hamiltonian that couples $\vec{v}$ and $\vec{w}$ states, $U_c$ denote the Clifford transformation that is used for tapering, and subscript $_t$ denote tapered versions of stuff.

The following matrix elements should be equal

\begin{align}
      & \langle \phi|H|\psi \rangle\\
    = & \langle \phi|H(\vec{v}, \vec{w})|\psi \rangle\\
    = & \langle \vec{v}, \phi_t | U_c H U_c^\dagger | \vec{w}, \psi_t \rangle \\
    = & \langle \vec{v}, \phi_t | U_c H(\vec{v}, \vec{w}) U_c^\dagger | \vec{w}, \psi_t \rangle \\
    = & \langle \phi_t| \Big(U_c H(\vec{v}, \vec{w}) U_c^\dagger\Big)_t | \psi_t \rangle.
\end{align}

The purpose of the following cell is the verify all of these equalities.

In [ ]:
Nqubits    = 12
Nterms     = 1000

config_list = [
    [random.randint(0,1) for _ in range(Nqubits // 2)] for _ in range(10)
]

for v_config in config_list:
    for w_config in config_list
        # generate Clifford transformation 

        Uc_wo_reordering = seniority_solving_clifford_operator(Nqubits)

        # generate random Hamiltonian, process it in the four separate ways given in the above sequence of equalities

        H                   = random_pauli_hamiltonian(Nqubits, Nterms)
        H_processed         = process_qubit_hamiltonian_to_remove_irrelevant_terms(H, Nqubits, v_config, w_config)
        H_rotated           = group_odds_and_evens(apply_unitary_product(H, Uc_wo_reordering), Nqubits)
        H_processed_rotated = group_odds_and_evens(apply_unitary_product(H_processed, Uc_wo_reordering), Nqubits)
        H_tapered           = process_qubit_hamiltonian_to_project_onto_symmetric_subspace(H, Nqubits, v_config, w_config)

        H_SPARSE                   = get_sparse_operator(H, Nqubits)
        H_processed_SPARSE         = get_sparse_operator(H_processed, Nqubits)
        H_rotated_SPARSE           = get_sparse_operator(H_rotated, Nqubits)
        H_processed_rotated_SPARSE = get_sparse_operator(H_processed_rotated, Nqubits)
        H_tapered_SPARSE           = get_sparse_operator(H_tapered, Nqubits // 2)

        # generate random bra and random ket. I need the full state, the rotated full state, and the tapered state

        v_string = ''.join(str(x) for x in v_config)
        w_string = ''.join(str(x) for x in w_config)

        ket_v                   = np.zeros(2 ** (Nqubits // 2))
        ket_v[int(v_string, 2)] = 1.0

        bra_N          = uniform(-1, 1, 2 ** (Nqubits // 2))
        bra_N          = bra_N / np.linalg.norm(bra_N)
        bra_2N         = decompress_state(bra_N, v_config)
        bra_2N_rotated = np.kron(ket_v, bra_N)

        ket_w                   = np.zeros(2 ** (Nqubits // 2))
        ket_w[int(w_string, 2)] = 1.0

        ket_N          = uniform(-1, 1, 2 ** (Nqubits // 2))
        ket_N          = ket_N / np.linalg.norm(ket_N)
        ket_2N         = decompress_state(ket_N, w_config)
        ket_2N_rotated = np.kron(ket_w, ket_N)

        # evaluate matrix elements 5 different ways

        me_1 = np.round(bra_2N @ H_SPARSE @ ket_2N, 6)
        me_2 = np.round(bra_2N @ H_processed_SPARSE @ ket_2N, 6)
        me_3 = np.round(bra_2N_rotated @ H_rotated_SPARSE @ ket_2N_rotated, 6)
        me_4 = np.round(bra_2N_rotated @ H_processed_rotated_SPARSE @ ket_2N_rotated, 6)
        me_5 = np.round(bra_N @ H_tapered_SPARSE @ ket_N, 6)
        vals = [me_1, me_2, me_3, me_4, me_5]

        print(all(x == vals[0] for x in vals), end='')
    
    print()

TrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrue
TrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrue
TrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrue
TrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrue
TrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrue
TrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrue
TrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrue
TrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrue
TrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrue
TrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrue
TrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrue
TrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrue
TrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrue
TrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrue
TrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrue
TrueTrueTrueTrueTrueTrueT

In [13]:
Nqubits = 14
Nterms  = 500

config_list = [
    [random.randint(0,1) for _ in range(Nqubits // 2)] for _ in range(10)
]

for v_config in config_list:
    for w_config in config_list:

        # Clifford transformation

        Uc_wo_reordering = seniority_solving_clifford_operator(Nqubits)

        # Hamiltonian and processed versions

        H                   = random_pauli_hamiltonian(Nqubits, Nterms)
        H_processed         = process_qubit_hamiltonian_to_remove_irrelevant_terms(H, Nqubits, v_config, w_config)
        H_rotated           = group_odds_and_evens(apply_unitary_product(H, Uc_wo_reordering), Nqubits)
        H_processed_rotated = group_odds_and_evens(apply_unitary_product(H_processed, Uc_wo_reordering), Nqubits)
        H_tapered           = process_qubit_hamiltonian_to_project_onto_symmetric_subspace(H, Nqubits, v_config, w_config)

        H_SPARSE                   = get_sparse_operator(H, Nqubits)
        H_processed_SPARSE         = get_sparse_operator(H_processed, Nqubits)
        H_rotated_SPARSE           = get_sparse_operator(H_rotated, Nqubits)
        H_processed_rotated_SPARSE = get_sparse_operator(H_processed_rotated, Nqubits)
        H_tapered_SPARSE           = get_sparse_operator(H_tapered, Nqubits // 2)

        # generate states

        v_string = ''.join(str(z) for z in v_config)
        w_string = ''.join(str(z) for z in w_config)

        v_state = np.zeros(2 ** (Nqubits // 2))
        v_state[int(v_string, 2)] = 1.0

        w_state = np.zeros(2 ** (Nqubits // 2))
        w_state[int(w_string, 2)] = 1.0

        ket_t = uniform(-1, 1, 2 ** (Nqubits // 2))
        ket_t = ket_t / np.linalg.norm(ket_t)
        ket   = decompress_state(ket_t, w_config)
        ket_r = np.kron(w_state, ket_t)

        bra_t = uniform(-1, 1, 2 ** (Nqubits // 2))
        bra_t = bra_t / np.linalg.norm(bra_t)
        bra   = decompress_state(bra_t, v_config)
        bra_r = np.kron(v_state, bra_t)

        # evaluate matrix elements

        me1  = np.round(bra @ H_SPARSE @ ket, 6)
        me2  = np.round(bra @ H_processed_SPARSE @ ket, 6)
        me3  = np.round(bra_r @ H_rotated_SPARSE @ ket_r, 6)
        me4  = np.round(bra_r @ H_processed_rotated_SPARSE @ ket_r, 6)
        me5  = np.round(bra_t @ H_tapered_SPARSE @ ket_t, 6)
        vals = [me1, me2, me3, me4, me5]

        print(all(x == vals[0] for x in vals), end='')

    print()

TrueTrueTrueTrueTrueTrueTrueTrueTrueTrue
TrueTrueTrueTrueTrueTrueTrueTrueTrueTrue
TrueTrueTrueTrueTrueTrueTrueTrueTrueTrue
TrueTrueTrueTrueTrueTrueTrueTrueTrueTrue
TrueTrueTrueTrueTrueTrueTrueTrueTrueTrue
TrueTrueTrueTrueTrueTrueTrueTrueTrueTrue
TrueTrueTrueTrueTrueTrueTrueTrueTrueTrue
TrueTrueTrueTrueTrueTrueTrueTrueTrueTrue
TrueTrueTrueTrueTrueTrueTrueTrueTrueTrue
TrueTrueTrueTrueTrueTrueTrueTrueTrueTrue
